# Google Colab (Skip locally)

In [ ]:
#@title Mount your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
%matplotlib inline
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@title Link your assignment folder & install requirements
#@markdown Enter the path to the assignment folder in your Google Drive
# If you run this notebook locally or on a cluster (i.e. not on Google Colab)
# you can delete this cell which is specific to Google Colab. You may also
# change the paths for data/logs in Arguments below.
import sys
import os
import shutil
import warnings

folder = "/content/gdrive/MyDrive/IFT6135/HW2_2025/src" #@param {type:"string"}
!ln -Ts "$folder" /content/src 2> /dev/null

# Add the assignment folder to Python path
if '/content/src' not in sys.path:
  sys.path.insert(0, '/content/src')

# Check if CUDA is available
import torch
if not torch.cuda.is_available():
  warnings.warn('CUDA is not available.')

# Imports & Setup

In [ ]:
%matplotlib inline
import warnings

from main import train_model, run_attacks
from shutil import make_archive

from visualisation.AttacksVisualisation import (
    plot_attack_effect_on_accuracy,
    plot_training_curves,
    plot_transfer_attack_effects
)
from visualisation.TrainingVisualisation import (
    plot_training_result,
    plot_models_together,
)
from arguments import Arguments
from models.classical.LeNet import LeNet
from models.classical.MLP import MLP
from models.classical.ResNet18 import ResNet18
from models.classical.MLPMixer import MLPMixer
from models.hybrid.HQNN_Quanv import HQNN_Quanv
from models.hybrid.HQNN_Parallel import HQNN_Parallel

from LoggingService import LoggingService

# Experiment #1

## Experiment Setup

In [2]:
# Define a arguments and set global defaults
LoggingService("experiment1")
seeds = [0, 42]
args = Arguments()
args.verbose = False
args.n_steps = 100
args.num_workers = 0
args.train_batch_size = 512
args.eval_batch_size = 512

# 2) Define your list of model‐configurations: (name, class, kwargs)
model_specs = [
    # ("ResNet18_1", ResNet18, {
    #     "num_classes": 10
    # }),
    # ("MLPMixer_1", MLPMixer, {
    #     "num_classes": 10,
    #     "img_size": 32,
    #     "patch_size": 4,
    #     "embed_dim": 256,
    #     "num_blocks": 4,
    #     "drop_rate": 0.0,
    #     "activation": "gelu",
    #     "mlp_ratio": [1, 1]
    # }),
    # ("MLP_1", MLP, {
    #     "input_size": 3072,
    #     "hidden_sizes": [1024, 512, 64, 64],
    #     "num_classes": 10,
    #     "activation": "relu"
    # }),
    ("HQNN_Parallel_1", HQNN_Parallel, {
        "in_channels": 1,
        "classes": 10
    }),
    ("LeNet_1", LeNet, {
        "in_channels": 1,
        "classes": 10
    }),
]

Tensorboard path: ../logs\experiment1\2025-06-10@11-36-18/tensorboard


## Models Training

In [3]:
# Train each model spec across all seeds, collect results as (model, seed)
trained_models = {}
for (model_name, model_cls, config) in model_specs:
    args.exp_name = model_name
    args.model = model_cls
    args.model_config = config

    trained_models[model_name] = train_model(args, seeds)

>>> Starting experiment 'HQNN_Parallel_1' over seeds [0, 42]
(1/2); Seed = 0  ...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

   | Name       | Type             | Params | Mode 
---------------------------------------------------------
0  | conv1      | Conv2d           | 416    | train
1  | batchNorm1 | BatchNorm2d      | 32     | train
2  | relu1      | ReLU             | 0      | train
3  | maxpool1   | MaxPool2d        | 0      | train
4  | conv2      | Conv2d           | 12.8 K | train
5  | batchNorm2 | BatchNorm2d      | 64     | train
6  | relu2      | ReLU             | 0      | train
7  | maxpool2   | MaxPool2d        | 0      | train
8  | fc1        | Linear           | 31.4 K | train
9  | relu3      | ReLU             | 0      | train
10 | qlayer1    | ParallelVQC      | 60     | train
11 | fc2        | Linear           | 210    | train
12 | logSoftmax | LogSoftmax       | 0      | train
13 | loss_fn    | CrossEntropyLoss | 0      | train
--------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\fwilh\.conda\envs\QAML\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\fwilh\.conda\envs\QAML\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\fwilh\.conda\envs\QAML\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\Users\fwilh\.conda\envs\QAML\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.


## Attacks

In [4]:
# Run transfer attacks
attacks_metrics = run_attacks(trained_models, args)


>>> Running attacks for seed : 0

--- Attacking with model 'HQNN_Parallel_1' (seed 0) ---
Clean accuracy before generating adversarial examples: 0.2048


Computing gradients for HQNN_Parallel_1:   2%|▏         | 151/10000 [00:41<44:56,  3.65it/s]


Clean accuracy after generating adversarial examples: 0.2048

--- Attacking with model 'LeNet_1' (seed 0) ---
Clean accuracy before generating adversarial examples: 0.9460


Computing gradients for LeNet_1:   2%|▏         | 151/10000 [00:01<01:24, 116.93it/s]


Clean accuracy after generating adversarial examples: 0.9460

>>> Running attacks for seed : 42

--- Attacking with model 'HQNN_Parallel_1' (seed 42) ---
Clean accuracy before generating adversarial examples: 0.1485


Computing gradients for HQNN_Parallel_1:   2%|▏         | 151/10000 [00:42<45:54,  3.58it/s]


Clean accuracy after generating adversarial examples: 0.1485

--- Attacking with model 'LeNet_1' (seed 42) ---
Clean accuracy before generating adversarial examples: 0.9541


Computing gradients for LeNet_1:   2%|▏         | 151/10000 [00:01<01:10, 139.15it/s]


Clean accuracy after generating adversarial examples: 0.9541


In [16]:
from pprint import pprint

pprint(trained_models['HQNN_Parallel_1'][0].all_metrics, depth=2)
metrics = trained_models['HQNN_Parallel_1'][0].all_metrics

print("Training metrics sizes:")
print("loss:", len(metrics['train']['loss']))
print("acc:", len(metrics['train']['acc']))
print("step:", len(metrics['train']['step']))
print("epoch:", len(metrics['train']['epoch']))

print("\nValidation metrics sizes:")
print("loss:", len(metrics['val']['loss']))
print("acc:", len(metrics['val']['acc']))
print("step:", len(metrics['val']['step']))
print("epoch:", len(metrics['val']['epoch']))

{'train': {'acc': [...], 'epoch': [...], 'loss': [...], 'step': [...]},
 'val': {'acc': [...], 'epoch': [...], 'loss': [...], 'step': [...]}}
Training metrics sizes:
loss: 100
acc: 100
step: 100
epoch: 100

Validation metrics sizes:
loss: 26
acc: 26
step: 26
epoch: 26


## Figures

In [ ]:
plot_training_curves(trained_models,
                    title_prefix="training")

plot_transfer_attack_effects(attacks_metrics,
                            title_prefix="transfer")


In [ ]:
make_archive('experiment1', 'zip', 'logs/experiment1')